# RealtyScope Phase 4 EDA With Observation History

Notebook này dùng dữ liệu đã persist trong PostgreSQL để nối latest state từ `listings` với lịch sử `listing_observations`. Mục tiêu là chuyển Phase 4 từ data readiness sang EDA có evidence cho OSM enrichment và baseline ML.

Các section bắt buộc: latest listing distributions, observation count and price-change analysis, coordinate coverage, candidate OSM enrichment readiness, naive baseline target distribution, và Kết luận tiếng Việt.


## Setup

Notebook đọc `DATABASE_URL` từ environment nếu có; nếu không, dùng settings mặc định của RealtyScope. Trước khi chạy, đảm bảo Alembic migration đã ở `head` và database chứa batch Domclick đã persist.


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sqlalchemy import text

from realtyscope.database.session import create_database_engine

DATABASE_URL = os.environ.get("DATABASE_URL")
engine = create_database_engine(DATABASE_URL)
sns.set_theme(style="whitegrid")
DATABASE_URL or "realtyscope settings database_url"

## Load latest listings and observation history

Các query bên dưới cố ý đọc từ bảng `listings` và `listing_observations`, không đọc JSONL capture artifact. Điều này giữ EDA bám vào persistence contract của Phase 3.


In [ ]:
with engine.connect() as connection:
    listings = pd.read_sql_query(
        text(
            """
            SELECT id, city, address_text, latitude, longitude, price_rub,
                   total_area_m2, rooms, floor, floors_total, building_year,
                   property_type, has_coordinates, is_ml_ready, active,
                   first_seen_at, last_seen_at
            FROM listings
            ORDER BY id
            """
        ),
        connection,
    )
    observations = pd.read_sql_query(
        text(
            """
            SELECT listing_id, source_listing_id, observed_at, price_rub,
                   price_per_m2, total_area_m2, rooms, floor, floors_total,
                   active, status
            FROM listing_observations
            ORDER BY listing_id, observed_at
            """
        ),
        connection,
    )

listings["price_per_m2"] = listings["price_rub"] / listings["total_area_m2"]
observations["observed_at"] = pd.to_datetime(observations["observed_at"], errors="coerce")
listings.shape, observations.shape

## latest listing distributions

This section profiles the current canonical listing state: `price_rub`, `price_per_m2`, area, rooms, floor and candidate outliers. It is the safest source for cross-sectional EDA because each listing has one latest row.


In [ ]:
distribution_columns = [
    "price_rub",
    "price_per_m2",
    "total_area_m2",
    "rooms",
    "floor",
    "floors_total",
]
latest_distribution_summary = (
    listings[distribution_columns].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T
)
rooms_distribution = listings["rooms"].value_counts(dropna=False).sort_index()
outlier_candidates = listings.nlargest(10, "price_rub")[
    ["id", "price_rub", "price_per_m2", "total_area_m2", "rooms", "floor", "floors_total"]
]
latest_distribution_summary, rooms_distribution, outlier_candidates

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(listings["price_rub"], bins=40, ax=axes[0])
axes[0].set_title("price_rub distribution")
sns.histplot(listings["price_per_m2"], bins=40, ax=axes[1])
axes[1].set_title("price_per_m2 distribution")
sns.countplot(data=listings, x="rooms", ax=axes[2])
axes[2].set_title("rooms distribution")
plt.tight_layout()

## observation count and price-change analysis

This section checks whether `listing_observations` already contains temporal signal. With one observation per listing, we can validate persistence coverage but should not claim market trends or price-change dynamics yet.


In [ ]:
observation_counts = observations.groupby("listing_id").size().rename("observation_count")
distinct_prices = (
    observations.groupby("listing_id")["price_rub"].nunique().rename("distinct_prices")
)
observation_profile = pd.concat([observation_counts, distinct_prices], axis=1).reset_index()
price_change_examples = observation_profile.loc[observation_profile["distinct_prices"] > 1].head(20)
observation_summary = {
    "observations_total": int(len(observations)),
    "listings_with_observations": int(observation_counts.size),
    "listings_with_multiple_observations": int((observation_counts > 1).sum()),
    "listings_with_price_changes": int((distinct_prices > 1).sum()),
}
observation_summary, observation_profile["observation_count"].describe(), price_change_examples

## coordinate coverage

Coordinates are the gate for neighborhood features and future OpenStreetMap enrichment. This section checks whether `latitude`, `longitude`, and `has_coordinates` agree.


In [ ]:
coordinate_complete = (
    listings["has_coordinates"] & listings["latitude"].notna() & listings["longitude"].notna()
)
coordinate_summary = {
    "listings_total": int(len(listings)),
    "with_coordinates": int(coordinate_complete.sum()),
    "without_coordinates": int((~coordinate_complete).sum()),
    "coverage": float(coordinate_complete.mean()) if len(listings) else 0.0,
}
coordinate_bounds = listings.loc[coordinate_complete, ["latitude", "longitude"]].agg(["min", "max"])
coordinate_summary, coordinate_bounds

## candidate OSM enrichment readiness

OpenStreetMap enrichment can start only after coordinate coverage is stable. Phase 4.1 does not call OSM or Overpass; it only checks readiness for future fixture-tested features such as transport, schools, parks, shops and healthcare counts around each listing. Any future OSM-derived docs/UI must include OpenStreetMap attribution.


In [ ]:
osm_candidate_features = [
    "transport_count_500m",
    "transport_count_1000m",
    "nearest_transport_m",
    "schools_count_1000m",
    "parks_count_1000m",
    "shops_count_1000m",
    "healthcare_count_1000m",
]
osm_readiness = {
    "coordinate_ready_rows": coordinate_summary["with_coordinates"],
    "coordinate_coverage": coordinate_summary["coverage"],
    "candidate_features": osm_candidate_features,
    "live_osm_called": False,
}
osm_readiness

## naive baseline target distribution

Before model training, inspect the target distribution and define a naive baseline candidate. This notebook only prepares evidence; actual model training and MLflow logging belong to later Phase 4 tasks.


In [ ]:
target_summary = (
    listings[["price_rub", "price_per_m2"]].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T
)
naive_baselines = {
    "median_price_rub": float(listings["price_rub"].median()) if not listings.empty else None,
    "median_price_per_m2": float(listings["price_per_m2"].median()) if not listings.empty else None,
}
target_summary, naive_baselines

## Kết luận tiếng Việt

Dữ liệu hiện tại đủ tốt để bắt đầu EDA và baseline ML đơn giản: 2000 listing đều có tọa độ và đều ML-ready theo readiness audit. Tuy nhiên `listing_observations` hiện mới có 2000 observation cho 2000 listing, tức mỗi listing chỉ có một snapshot; vì vậy chưa nên kết luận xu hướng thời gian hoặc biến động giá.

Ưu tiên tiếp theo: giữ daily capture chạy ổn để tạo nhiều observation/listing, thêm OSM enrichment bằng fixture tests trước khi gọi live Overpass, và dùng baseline naive theo median `price_rub` hoặc median `price_per_m2` như mốc so sánh ban đầu.
